# 1-1절 연습 문제 풀이

이 노트북은 1-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch01/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 1-1

파이토치 프레임워크를 통해 딥러닝 모델 개발에 사용할 수 있는 여러 데이터셋을 쉽게 구할 수 있다. 다음 코드는 MNIST 데이터셋을 불러와 images와 labels 두 텐서를 생성한다.

*코드 1-25 MNIST 데이터셋 불러오기*

```python
import torch
from torchvision import datasets, transforms
transform = transforms.ToTensor()
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=train_set, batch_size=32, shuffle=True)
images, labels = next(iter(train_loader))
```

이렇게 생성한 images와 labels 두 텐서의 형태를 출력해 보자.

In [ ]:
# MNIST 데이터셋을 불러와 한 배치를 꺼낸다.
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_set = datasets.MNIST(root='../../download', train=True,
                           download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=train_set, batch_size=32,
                                           shuffle=True)
images, labels = next(iter(train_loader))

print(f'images 텐서의 형태: {tuple(images.shape)}')   # 출력값: (32, 1, 28, 28)
print(f'labels 텐서의 형태: {tuple(labels.shape)}')   # 출력값: (32,)

images는 (배치 크기, 색상 채널, 세로, 가로) 순서의 4차원 텐서이고, labels는 배치 크기만큼의 정답을 담은 1차원 텐서다.

> 책의 `root='./data'`를 저장소 구조에 맞춰 `'../../download'`로 바꿔 사용했다.

## 연습 1-2

[연습 문제 1-1]에서 생성한 텐서 중 images 텐서에는 가로, 세로 각각 28픽셀로 구성된 32장의 이미지 정보가 저장되어 있다. 이 텐서의 두 번째 차원은 컬러 이미지의 색상별 데이터로 구성된 색상 채널 차원인데, MNIST 데이터와 같이 색상 채널이 하나뿐인 회색조 이미지에서는 색상 채널 차원이 불필요하다. images 텐서에서 색상 채널 차원을 제거해 보자.

In [ ]:
# 크기가 1인 색상 채널 차원(두 번째 차원) 제거
images_2d = images.squeeze(1)          # (32, 1, 28, 28) -> (32, 28, 28)
print(f'squeeze(1): {tuple(images_2d.shape)}')

# 인자 없이 호출하면 크기가 1인 모든 차원을 제거하므로 결과는 같다.
print(f'squeeze():  {tuple(images.squeeze().shape)}')

색상 채널은 크기가 1인 차원이므로 `squeeze()`로 제거한다. 배치 크기가 1이 되는 경우까지 고려하면 제거할 차원을 명시한 `squeeze(1)`이 더 안전하다.

## 연습 1-3

[연습 문제 1-2]에서 색상 채널을 제거한 텐서의 마지막 두 차원은 각각 가로 픽셀과 세로 픽셀에 해당하는 차원이다. 이 두 차원의 데이터를 이어 붙여 각 이미지가 크기 784의 1차원 텐서로 표현되도록 images 텐서를 변형해 보자.

In [ ]:
# 마지막 두 차원(28, 28)을 하나로 이어 붙여 784 크기의 벡터로 변형
images_flat = images_2d.view(images_2d.size(0), -1)   # (32, 28, 28) -> (32, 784)
print(f'view:    {tuple(images_flat.shape)}')

# flatten()으로 시작 차원을 지정해도 결과는 같다.
print(f'flatten: {tuple(images_2d.flatten(start_dim=1).shape)}')
print(f'요소 수 보존: {images_2d.numel() == images_flat.numel()}')

`-1`을 지정하면 남은 요소 수에 맞춰 크기가 자동으로 계산된다(28 × 28 = 784).

## 연습 1-4

다음은 세 명의 학생이 네 과목의 시험을 두 번 치른 후 성적을 정리한 표이다.

| 학생 | 1회차 | 2회차 |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
|  | 국어 | 영어 | 수학 | 과학 | 국어 | 영어 | 수학 | 과학 |
| A | 55 | 48 | 97 | 97 | 60 | 5 | 82 | 75 |
| B | 62 | 36 | 92 | 96 | 66 | 47 | 38 | 65 |
| C | 73 | 20 | 56 | 100 | 49 | 54 | 43 | 49 |

이 표의 데이터로 학생 - 회차 - 과목 순의 텐서를 만들어 보자.

생성한 텐서를 permute() 메서드를 사용해 차원이 과목 - 회차 - 학생 순서인 텐서로 변형해 보자.

In [ ]:
# 학생 - 회차 - 과목 순서의 (3, 2, 4) 텐서
scores = torch.tensor([
    [[55, 48, 97, 97], [60,  5, 82, 75]],   # 학생 A - 1회차, 2회차
    [[62, 36, 92, 96], [66, 47, 38, 65]],   # 학생 B
    [[73, 20, 56, 100], [49, 54, 43, 49]],  # 학생 C
])
print(f'학생-회차-과목: {tuple(scores.shape)}')

# 과목 - 회차 - 학생 순서로 차원 재배치 (0:학생, 1:회차, 2:과목 -> 2, 1, 0)
permuted = scores.permute(2, 1, 0)
print(f'과목-회차-학생: {tuple(permuted.shape)}')

# 값이 제대로 옮겨졌는지 확인: 학생 C(2)의 2회차(1) 수학(2) 점수
print(f'원본:   {scores[2, 1, 2].item()}')
print(f'재배치: {permuted[2, 1, 2].item()}')

`permute()`는 모든 차원의 새 순서를 빠짐없이 지정해야 한다. 학생(0) - 회차(1) - 과목(2)을 과목 - 회차 - 학생으로 바꾸려면 `permute(2, 1, 0)`이다.

## 연습 1-5

다음은 미래에 있을지도 모르는 어떤 상황에 대한 설명이다.

기술이 크게 발전한 인류는 지구와 모든 면에서 비슷한 외계 행성을 발견했다. 이주 가능 여부를 확인하려고 파견된 탐사대의 임무에는, 갈릴레오가 발견하고 뉴턴이 법칙으로 정리한 다음 공식이 외계 행성에서도 성립하는지 확인하는 실험이 들어 있었다.

낙하거리 = 1/2 x 중력가속도 x 낙하시간2

탐사대는 임의의 높이에서 물체를 50번 떨어뜨려 낙하시간을 측정한 후 지구로 데이터를 보내왔다. 지구의 분석팀은 이 데이터를 바탕으로 이 외계 행성에서도 지구와 같은 물리 법칙이 성립하는지 확인해야 한다.

물론 현재의 과학기술로는 불가능한 일이므로, 이런 미래를 시뮬레이션하려면 그럴싸한 데이터를 직접 생성해 사용해야 한다. 다음 조건에 맞는 두 개의 텐서를 만들어 보자.

관측시간 텐서: 0~10 범위의 실수형 난수로 구성된 (50, 1) 형태의 텐서

관측거리 텐서: 다음 순서로 구성된 (50, 1) 형태의 텐서 - ① 관측시간 x에 대해 4.9x2로 낙하거리를 계산 ② 표준정규분포를 따르는 난수에 낙하거리의 10%를 곱해 노이즈를 생성(노이즈는 관측 오차를 흉내 낸 것이다.) ③ 낙하거리에 노이즈를 더한 값을 관측거리로 사용

단, 관측시간 텐서의 요소와 관측거리 텐서의 요소는 서로 대응되어야 한다.

In [ ]:
# 관측시간: 0~10 범위의 실수형 난수 (50, 1)
GRAVITY = 4.9        # 낙하거리 = 1/2 x 중력가속도 x 낙하시간^2 의 계수
observed_time = torch.rand(50, 1) * 10

# 낙하거리 -> 노이즈 생성 -> 관측거리
fall_distance = GRAVITY * observed_time ** 2
noise = torch.randn(50, 1) * (fall_distance * 0.1)   # 낙하거리의 10% 크기
observed_distance = fall_distance + noise

print(f'관측시간 텐서: {tuple(observed_time.shape)}, '
      f'범위 {observed_time.min():.2f} ~ {observed_time.max():.2f}')
print(f'관측거리 텐서: {tuple(observed_distance.shape)}')
print(f'첫 세 쌍(시간, 거리):')
for t, d in zip(observed_time[:3], observed_distance[:3]):
    print(f'  {t.item():5.2f}초 -> {d.item():7.2f}m')

두 텐서를 같은 순서로 만들었으므로 같은 인덱스의 요소가 서로 대응한다. 노이즈를 낙하거리에 비례하도록 만들면 거리가 클수록 관측 오차도 커지는 상황을 흉내 낼 수 있다.

## 연습 1-6

[도전 문제] 다음 두 텐서의 덧셈 연산에서 파이토치가 수행하는 브로드캐스팅 과정을 직접 구현해 보자. 브로드캐스팅을 직접 구현한 결과와 파이토치의 브로드캐스팅 연산 결과가 같은지 확인해 보자.

*코드 1-26 형태가 다른 두 텐서의 덧셈*

```python
op1 = torch.tensor([[[1, 2]], [[3, 4]]])
op2 = torch.tensor([[5, 6], [7, 8]])
tensor_sum = op1 + op2
print(f'연산 결과: {tensor_sum}')
```

힌트: 브로드캐스팅 1단계는 차원을 추가하는 메서드를 사용하면 되며, 브로드캐스팅 2단계는 여러 텐서를 이어 붙이는 메서드를 사용하면 된다.

In [ ]:
a = torch.tensor([[1], [2], [3]])       # (3, 1)
b = torch.tensor([10, 20, 30, 40])     # (4,)

# 1) 차원 수 맞추기: 앞쪽에 크기 1인 차원을 추가해 (1, 4)로 만든다.
b_expanded = b.unsqueeze(0)                       # (4,) -> (1, 4)
# 2) 크기가 1인 차원을 상대 텐서의 크기만큼 복제한다.
a_broadcast = a.repeat(1, b_expanded.size(1))     # (3, 1) -> (3, 4)
b_broadcast = b_expanded.repeat(a.size(0), 1)     # (1, 4) -> (3, 4)
# 3) 같은 형태가 되었으므로 요소별로 더한다.
manual = a_broadcast + b_broadcast

auto = a + b        # 파이토치의 브로드캐스팅
print(manual)
print(f'파이토치 연산과 동일한가: {torch.equal(manual, auto)}')

브로드캐스팅은 ① 뒤쪽 차원부터 맞춰 보며 부족한 차원을 앞에 추가하고, ② 크기가 1인 차원을 상대 크기만큼 늘린 뒤, ③ 요소별로 연산하는 과정이다. `repeat()`로 실제 복제해 구현하면 그 과정을 눈으로 확인할 수 있다.